# Ingest data into Unity Catalog

## Industry scenario: GreenGrid Energy

**GreenGrid Energy** is a renewable energy company operating wind farms and solar parks across Europe and North America. Their data engineering team needs to ingest data from multiple sources into Unity Catalog:

- **Turbine sensor readings** — IoT devices producing continuous power output and weather metrics
- **Operational event feeds** — Change data from SCADA systems tracking turbine status transitions
- **Market price files** — Hourly energy spot prices delivered as CSV batches to cloud storage
- **Maintenance records** — Uploaded as JSON files by field crews after each service visit

This notebook walks through each ingestion technique covered in the module using GreenGrid data as the running example.

### Setup

The setup script creates the `trainer_demo.demo_07` schema with the following tables the demos reference:

| Table | Description |
|---|---|
| `energy_sites` | Wind farm and solar park locations (20 sites) |
| `turbines` | Individual turbine / panel assets (200 records) |
| `turbine_readings` | Historical hourly power output readings (50,000 rows) |
| `turbine_events` | CDC-style change feed — turbine status transitions |
| `market_prices` | Hourly energy spot prices by region |

Run the cell below to initialise all tables.

In [ ]:
from scripts.setup import setup
setup(spark)

## Ingest data with Lakeflow Connect

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-lakeflow-connect.png)

### Demo: Configure a Lakeflow Connect ingestion pipeline

> **Trainer note:** Lakeflow Connect is configured through the Azure Databricks UI or via Asset Bundles — there is no runnable notebook code. Walk through the UI wizard or show the YAML definition below.

The following Asset Bundle configuration ingests the `turbine_readings` table from an on-premises SQL Server SCADA system into Unity Catalog. It uses **SCD Type 2** to preserve the full history of turbine status changes, and **CDC** to track only modified rows between runs.

```yaml
resources:
  pipelines:
    gateway_greengrid:
      name: greengrid-scada-gateway
      gateway_definition:
        connection_name: scada-sql-server-prod
        gateway_storage_catalog: trainer_demo
        gateway_storage_schema: demo_07
        gateway_storage_name: greengrid_gateway

    pipeline_turbine_readings:
      name: greengrid-turbine-readings-pipeline
      catalog: trainer_demo
      schema: demo_07
      ingestion_definition:
        ingestion_gateway_id: ${resources.pipelines.gateway_greengrid.id}
        objects:
          - table:
              source_schema: dbo
              source_table: turbine_readings
              destination_catalog: trainer_demo
              destination_schema: demo_07
              table_configuration:
                scd_type: SCD_TYPE_2
                sequence_by: reading_ts
                include_columns:
                  - turbine_id
                  - reading_ts
                  - power_output_kw
                  - wind_speed_ms
                  - operational_status
```

Key points to highlight:
- The **ingestion gateway** continuously reads change data from the SQL Server source
- `scd_type: SCD_TYPE_2` adds `__START_AT` and `__END_AT` columns to track row history
- `include_columns` limits data transfer — new source columns won't appear automatically

## Ingest data with notebooks

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-notebooks.png)

### Demo: Ingest turbine readings from files using DataFrames

GreenGrid's field teams periodically export turbine readings as CSV files to a Unity Catalog Volume. This demo shows how to read those files into a DataFrame and write them to a Delta table.

In [ ]:
# Write sample CSV data to a Unity Catalog Volume for the demo
# (Simulates files dropped by the field export process)

sample_csv = (
    "turbine_id,reading_ts,power_output_kw,wind_speed_ms,temperature_c,operational_status\n"
    "TRB-001,2024-03-01 06:00:00,1850.5,8.2,12.3,operational\n"
    "TRB-001,2024-03-01 07:00:00,2100.0,9.1,11.8,operational\n"
    "TRB-002,2024-03-01 06:00:00,0.0,2.1,10.5,offline\n"
    "TRB-002,2024-03-01 07:00:00,950.3,6.7,11.0,operational\n"
    "TRB-003,2024-03-01 06:00:00,3200.8,12.4,9.7,operational\n"
    "TRB-003,2024-03-01 07:00:00,3150.2,11.9,9.5,operational\n"
    "TRB-004,2024-03-01 06:00:00,1200.1,7.3,13.1,maintenance\n"
    "TRB-005,2024-03-01 06:00:00,2800.9,10.8,8.9,operational\n"
)

dbutils.fs.put(
    "/Volumes/trainer_demo/demo_07/sensor_data_landing/readings_2024_03_01.csv",
    sample_csv,
    overwrite=True
)
print("Sample CSV written to volume")

In [ ]:
# Read the CSV file into a DataFrame
df_readings = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("/Volumes/trainer_demo/demo_07/sensor_data_landing/readings_2024_03_01.csv")
)

df_readings.printSchema()
display(df_readings)

After reading the file, write it to a Unity Catalog Delta table.

The `mode` parameter controls how existing data is handled:
- `overwrite` — replaces all rows (suitable for a full refresh)
- `append` — adds rows to the existing table
- `ignore` — skips if the table already exists

In [ ]:
# Write to a Unity Catalog Delta table (append new batch)
df_readings.write.mode("append").saveAsTable("trainer_demo.demo_07.turbine_readings_batch")

spark.sql("SELECT * FROM trainer_demo.demo_07.turbine_readings_batch").display()

#### Reading semi-structured data with the VARIANT type

GreenGrid also receives maintenance JSON payloads where the structure varies by turbine model. The **VARIANT** type stores the raw JSON and lets you query individual fields without defining a fixed schema upfront.

In [ ]:
from pyspark.sql.functions import parse_json, variant_get, col

# Simulate JSON maintenance records with varying fields per turbine model
maintenance_json = spark.createDataFrame([
    ('{"turbine_id":"TRB-001","component":"gearbox","action":"lubrication","technician":"J.Smith","duration_hrs":2.5}',),
    ('{"turbine_id":"TRB-002","component":"blade","action":"inspection","remark":"surface erosion noted"}',),
], ["raw_payload"])

# Parse to VARIANT and extract specific fields
df_variant = maintenance_json.select(
    parse_json(col("raw_payload")).alias("data")
).select(
    variant_get(col("data"), "$.turbine_id",  "string").alias("turbine_id"),
    variant_get(col("data"), "$.component",   "string").alias("component"),
    variant_get(col("data"), "$.action",      "string").alias("action"),
    variant_get(col("data"), "$.technician",  "string").alias("technician"),  # nullable
)

display(df_variant)

## Ingest data with SQL methods

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-sql-methods.png)

### Demo: Ingest and refresh tables with SQL methods

SQL commands offer a declarative, readable approach to ingestion. This demo shows three complementary techniques using the GreenGrid turbine data.

#### CTAS — Create a bronze layer from the production readings table

In [ ]:
-- CTAS: combine table creation and data population in one statement
-- Useful for initial loads and one-time schema migrations

CREATE TABLE IF NOT EXISTS trainer_demo.demo_07.bronze_turbine_readings
AS
SELECT
    turbine_id,
    reading_ts,
    ROUND(power_output_kw, 2)    AS power_output_kw,
    wind_speed_ms,
    UPPER(operational_status)   AS operational_status,
    CURRENT_TIMESTAMP()         AS ingested_at
FROM trainer_demo.demo_07.turbine_readings
WHERE operational_status != 'decommissioned';

SELECT COUNT(*) AS row_count FROM trainer_demo.demo_07.bronze_turbine_readings;

#### CREATE OR REPLACE TABLE — Refresh aggregated daily metrics

Use this when you want to completely replace a derived table. Unlike dropping and recreating, `CREATE OR REPLACE` preserves granted permissions and row filters.

In [ ]:
-- CREATE OR REPLACE TABLE: full refresh of an aggregated metrics table
-- Replaces all existing rows atomically; permissions and policies are preserved

CREATE OR REPLACE TABLE trainer_demo.demo_07.daily_site_production
AS
SELECT
    t.site_id,
    s.site_name,
    s.energy_type,
    DATE(r.reading_ts)                  AS production_date,
    SUM(r.power_output_kw)              AS total_output_kw,
    AVG(r.power_output_kw)              AS avg_output_kw,
    COUNT(DISTINCT r.turbine_id)        AS active_turbines
FROM trainer_demo.demo_07.turbine_readings r
JOIN trainer_demo.demo_07.turbines       t  ON r.turbine_id = t.turbine_id
JOIN trainer_demo.demo_07.energy_sites   s  ON t.site_id    = s.site_id
WHERE r.operational_status = 'operational'
GROUP BY t.site_id, s.site_name, s.energy_type, DATE(r.reading_ts);

SELECT * FROM trainer_demo.demo_07.daily_site_production
ORDER BY production_date DESC
LIMIT 10;

#### COPY INTO — Incrementally load new CSV files from a volume

`COPY INTO` is idempotent: files already loaded are tracked and skipped automatically, making it safe to run on a schedule even as the source directory accumulates files.

In [ ]:
-- Ensure the target table exists with the expected schema
CREATE TABLE IF NOT EXISTS trainer_demo.demo_07.market_prices_bronze (
    price_id        BIGINT,
    region          STRING,
    price_date      DATE,
    hour_utc        INT,
    spot_price_eur  DOUBLE,
    currency        STRING,
    source_file     STRING
);

-- COPY INTO: load CSV files from the volume (idempotent — already-loaded files are skipped)
COPY INTO trainer_demo.demo_07.market_prices_bronze
FROM (
    SELECT
        price_id,
        region,
        CAST(price_date AS DATE)        AS price_date,
        CAST(hour_utc AS INT)           AS hour_utc,
        CAST(spot_price_eur AS DOUBLE)  AS spot_price_eur,
        currency,
        _metadata.file_name             AS source_file
    FROM '/Volumes/trainer_demo/demo_07/sensor_data_landing/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS  ('mergeSchema' = 'true');

SELECT COUNT(*) AS rows_loaded FROM trainer_demo.demo_07.market_prices_bronze;

## Ingest data with CDC feed

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-cdc-feed.png)

### Demo: Ingest operational status changes with the AUTO CDC API

GreenGrid's SCADA system emits a continuous feed of turbine status changes as structured change records. Each record contains the turbine data, the operation type (`INSERT`, `UPDATE`, `DELETE`), and a sequence number for ordering.

The **AUTO CDC API** in Lakeflow Spark Declarative Pipelines handles deduplication, out-of-order event handling, and SCD logic automatically.

> **Trainer note:** `dp.create_auto_cdc_flow` runs inside a Lakeflow Pipeline definition. The cells below illustrate the CDC patterns using a MERGE-based approach that can be run directly in a notebook to show the before/after result.

In [ ]:
# Simulate a CDC change feed from the SCADA system
# Each row represents a change event captured from the source database

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

cdc_schema = StructType([
    StructField("operation",          StringType(),   False),  # INSERT / UPDATE / DELETE
    StructField("sequence_num",       IntegerType(),  False),  # Monotonically increasing
    StructField("turbine_id",         StringType(),   True),
    StructField("site_id",            StringType(),   True),
    StructField("capacity_kw",        DoubleType(),   True),
    StructField("operational_status", StringType(),   True),
    StructField("last_updated_ts",    StringType(),   True),
])

cdc_changes = [
    Row("INSERT", 1, "TRB-901", "SITE-01", 3500.0, "commissioning", "2024-03-01 08:00:00"),
    Row("INSERT", 2, "TRB-902", "SITE-01", 3500.0, "commissioning", "2024-03-01 08:01:00"),
    Row("UPDATE", 3, "TRB-901", "SITE-01", 3500.0, "operational",   "2024-03-01 10:00:00"),
    Row("UPDATE", 4, "TRB-902", "SITE-01", 3500.0, "offline",       "2024-03-02 06:00:00"),
    Row("UPDATE", 5, "TRB-901", "SITE-01", 3500.0, "maintenance",   "2024-03-05 09:00:00"),
    Row("DELETE", 6, "TRB-902", "SITE-01", None,    None,           "2024-03-10 12:00:00"),
]

cdc_df = spark.createDataFrame(cdc_changes, schema=cdc_schema)
display(cdc_df)

#### SCD Type 1 — Keep only the current state

SCD Type 1 overwrites existing records when updates arrive. Only the most recent version of each turbine is stored — no historical rows.

In [ ]:
# Create the target table for SCD Type 1
spark.sql(
    "CREATE TABLE IF NOT EXISTS trainer_demo.demo_07.turbines_current ("
    "    turbine_id          STRING NOT NULL,"
    "    site_id             STRING,"
    "    capacity_kw         DOUBLE,"
    "    operational_status  STRING,"
    "    last_updated_ts     STRING"
    ")"
)

upserts = cdc_df.filter("operation != 'DELETE'").select(
    "turbine_id","site_id","capacity_kw","operational_status","last_updated_ts"
)
deletes = cdc_df.filter("operation = 'DELETE'").select("turbine_id")

upserts.createOrReplaceTempView("cdc_upserts")
deletes.createOrReplaceTempView("cdc_deletes")

spark.sql("""
    MERGE INTO trainer_demo.demo_07.turbines_current AS target
    USING cdc_upserts AS source
    ON target.turbine_id = source.turbine_id
    WHEN MATCHED     THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("""
    MERGE INTO trainer_demo.demo_07.turbines_current AS target
    USING cdc_deletes AS source
    ON target.turbine_id = source.turbine_id
    WHEN MATCHED THEN DELETE
""")

spark.sql("SELECT * FROM trainer_demo.demo_07.turbines_current ORDER BY turbine_id").display()

#### SCD Type 2 — Preserve the full history of changes

SCD Type 2 adds `__START_AT` and `__END_AT` columns. When a turbine status changes, the old row is closed (end date set) and a new row is inserted. Analysts can reconstruct the exact state of each turbine at any point in time.

In [ ]:
# Create SCD Type 2 target table with history columns
spark.sql("""
    CREATE TABLE IF NOT EXISTS trainer_demo.demo_07.turbines_history (
        turbine_id          STRING NOT NULL,
        site_id             STRING,
        capacity_kw         DOUBLE,
        operational_status  STRING,
        last_updated_ts     STRING,
        __START_AT          STRING,
        __END_AT            STRING
    )
""")

# The AUTO CDC API (inside a Lakeflow Pipeline) implements this automatically:
#
#   dp.create_streaming_table('turbines_history')
#   dp.create_auto_cdc_flow(
#       target              = 'turbines_history',
#       source              = 'cdc_feed_view',
#       keys                = ['turbine_id'],
#       sequence_by         = col('sequence_num'),
#       apply_as_deletes    = col('operation') == 'DELETE',
#       except_column_list  = ['operation', 'sequence_num'],
#       stored_as_scd_type  = 2
#   )

# For demonstration, manually populate the SCD2 output:
history_rows = [
    ("TRB-901","SITE-01",3500.0,"commissioning","2024-03-01 08:00:00","2024-03-01 08:00:00","2024-03-01 10:00:00"),
    ("TRB-901","SITE-01",3500.0,"operational",  "2024-03-01 10:00:00","2024-03-01 10:00:00","2024-03-05 09:00:00"),
    ("TRB-901","SITE-01",3500.0,"maintenance",  "2024-03-05 09:00:00","2024-03-05 09:00:00",None),
    ("TRB-902","SITE-01",3500.0,"commissioning","2024-03-01 08:01:00","2024-03-01 08:01:00","2024-03-02 06:00:00"),
    ("TRB-902","SITE-01",3500.0,"offline",      "2024-03-02 06:00:00","2024-03-02 06:00:00","2024-03-10 12:00:00"),
]
history_df = spark.createDataFrame(
    history_rows,
    ["turbine_id","site_id","capacity_kw","operational_status","last_updated_ts","__START_AT","__END_AT"]
)
history_df.write.mode("append").saveAsTable("trainer_demo.demo_07.turbines_history")

spark.sql("""
    SELECT turbine_id, operational_status, __START_AT, __END_AT,
           CASE WHEN __END_AT IS NULL THEN 'current' ELSE 'historical' END AS row_type
    FROM trainer_demo.demo_07.turbines_history
    ORDER BY turbine_id, __START_AT
""").display()

## Ingest data with Spark Structured Streaming

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-spark-structured-streaming.png)

### Demo: Ingest real-time turbine sensor data with Spark Structured Streaming

GreenGrid's wind turbines emit a reading every few seconds. Structured Streaming captures these events continuously and lands them in a Delta table.

> **Trainer note:** We use the built-in `rate` source to simulate sensor events arriving in real time. In production, GreenGrid would connect to **Azure Event Hubs** via the Kafka-compatible interface.

In [ ]:
# Simulate real-time turbine sensor events using the built-in rate source
# The rate source generates (timestamp, value) rows at a configurable rate

from pyspark.sql.functions import rand, when, expr, round as spark_round

simulated_readings = (
    spark.readStream
        .format("rate")
        .option("rowsPerSecond", 5)
        .load()
        .withColumn(
            "turbine_id",
            expr("concat('TRB-', lpad(cast((value % 10) + 1 as string), 3, '0'))")
        )
        .withColumn("power_output_kw",    spark_round(rand() * 3500, 2))
        .withColumn("wind_speed_ms",      spark_round(rand() * 15 + 2, 1))
        .withColumn("temperature_c",      spark_round(rand() * 20 + 5, 1))
        .withColumn("operational_status", when(rand() > 0.95, "warning").otherwise("operational"))
        .select("timestamp","turbine_id","power_output_kw","wind_speed_ms","temperature_c","operational_status")
)

simulated_readings.printSchema()

In [ ]:
# Write the streaming data to a Delta table
# Each streaming job must have its own unique checkpoint location

checkpoint_path = "/Volumes/trainer_demo/demo_07/sensor_data_landing/checkpoints/turbine_stream"

query = (
    simulated_readings.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(processingTime="10 seconds")  # micro-batch every 10 s
        .toTable("trainer_demo.demo_07.turbine_stream_bronze")
)

print(f'Streaming query started: {query.id}')
print('Stream is running — run the next cell to check progress, then stop.')

In [ ]:
# Check stream status and recent data
print(query.status)

spark.sql("""
    SELECT COUNT(*) AS rows_ingested, MAX(timestamp) AS latest_ts
    FROM trainer_demo.demo_07.turbine_stream_bronze
""").display()

# Stop the stream once continuous ingestion has been demonstrated
query.stop()
print('Stream stopped.')

#### Connecting to Azure Event Hubs in production

In a production GreenGrid environment, turbines send readings to **Azure Event Hubs**. The Kafka-compatible interface uses SASL authentication. Store the connection string in **Databricks Secrets** — never hardcode credentials.

```python
eh_namespace      = "greengrid-telemetry"
eh_name           = "turbine-readings"
connection_string = dbutils.secrets.get(scope="eventhubs", key="connection-string")

kafka_options = {
    "kafka.bootstrap.servers": f"{eh_namespace}.servicebus.windows.net:9093",
    "subscribe": eh_name,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule '
        f'required username="$ConnectionString" password="{connection_string}";'
    ),
    "startingOffsets": "latest",
}

df_eh = spark.readStream.format('kafka').options(**kafka_options).load()
```

> **Key points:** Each streaming query needs its own unique checkpoint location. If a query restarts, it resumes from the last committed checkpoint — no data is lost or duplicated.

## Ingest data with Auto Loader

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-auto-loader.png)

### Demo: Incremental file ingestion with Auto Loader

Field crews upload maintenance report CSV files to a Unity Catalog Volume as they complete each service visit. Auto Loader watches the landing Volume and processes each file exactly once — even if the job is restarted.

In [ ]:
# Simulate three batches of maintenance report files arriving in the landing volume

import random
from datetime import datetime, timedelta

for batch_num in range(1, 4):
    rows = ["turbine_id,maintenance_date,service_type,technician,cost_eur,notes"]
    for i in range(5):
        tid  = f"TRB-{random.randint(1, 200):03d}"
        dt   = (datetime(2024, 3, batch_num) + timedelta(hours=i)).strftime('%Y-%m-%d %H:%M')
        svc  = random.choice(['blade_inspection','gearbox_oil','electrical_check','sensor_calibration'])
        tech = random.choice(['J.Smith','A.Patel','M.Koster','L.Nguyen'])
        cost = round(random.uniform(200, 2500), 2)
        rows.append(f'{tid},{dt},{svc},{tech},{cost},routine check')

    dbutils.fs.put(
        f"/Volumes/trainer_demo/demo_07/sensor_data_landing/maintenance_batch_{batch_num:02d}.csv",
        "\n".join(rows),
        overwrite=True
    )

print('3 maintenance batches written to the landing volume')
display(dbutils.fs.ls('/Volumes/trainer_demo/demo_07/sensor_data_landing/'))

In [ ]:
# Configure Auto Loader to incrementally ingest CSV files from the landing volume
# cloudFiles.format   — specifies the source file format
# cloudFiles.schemaLocation — stores the inferred schema for evolution tracking

base_path       = "/Volumes/trainer_demo/demo_07/sensor_data_landing"
schema_path     = f"{base_path}/.autoloader_schema"
checkpoint_path = f"{base_path}/.autoloader_checkpoint"

autoloader_query = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",          "csv")
        .option("cloudFiles.schemaLocation",   schema_path)
        .option("cloudFiles.inferColumnTypes", "true")  # detect date, numeric types
        .option("header",                      "true")
        .load(f"{base_path}/maintenance_batch_*.csv")  # pattern-match only maintenance files
    .writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema",         "true")          # allow schema evolution
        .trigger(availableNow=True)                     # process all pending files then stop
        .toTable("trainer_demo.demo_07.maintenance_bronze")
)

autoloader_query.awaitTermination()
print('Files processed:')
spark.sql("SELECT COUNT(*) AS total_rows FROM trainer_demo.demo_07.maintenance_bronze").display()

#### Schema evolution with Auto Loader

When field crews add new columns to their export (for example, a `gps_coordinates` field), Auto Loader detects the change and can evolve the target schema automatically using the `schemaEvolutionMode` option:

| Mode | Behaviour |
|---|---|
| `addNewColumns` *(default)* | Stream **fails** on new column, adds it to the schema, **restarts cleanly** |
| `rescue` | New columns land in `_rescued_data` — stream never fails |
| `failOnNewColumns` | Stream fails and does not restart until schema is manually updated |
| `none` | New columns are silently dropped |

```python
# Use rescue mode to prevent stream failures on ad-hoc column additions
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",             "csv")
    .option("cloudFiles.schemaLocation",     schema_path)
    .option("cloudFiles.schemaEvolutionMode","rescue")  # unknowns go to _rescued_data
    .option("header",                        "true")
    .load(base_path))
```

The `_rescued_data` column captures any data that doesn't match the expected schema — preventing data loss while keeping the pipeline stable.

## Ingest data with Lakeflow Spark Declarative Pipelines

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/ingest-data-into-unity-catalog.ingest-data-lakeflow-declarative-pipelines.png)

### Demo: Define ingestion pipelines with Lakeflow Spark Declarative Pipelines

Lakeflow Spark Declarative Pipelines let you declare the structure of your ingestion pipeline as code. The framework handles orchestration, retries, checkpointing, and exactly-once semantics automatically.

> **Trainer note:** The code below is **pipeline source code**. Create a new Pipeline in the Databricks UI, point it at a notebook or folder containing these definitions, and run it from the Pipelines interface — not directly from a notebook.

In [ ]:
# === PIPELINE SOURCE CODE (Python) ===
# Attach this notebook to a Lakeflow Pipeline — do not run cells directly

from pyspark import pipelines as dp
from pyspark.sql.functions import col

# ── Bronze: Auto Loader — new maintenance CSV files from the landing volume ──────
@dp.table
def maintenance_bronze():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format",         "csv")
            .option("cloudFiles.inferColumnTypes","true")
            .option("header",                    "true")
            .load("/Volumes/trainer_demo/demo_07/sensor_data_landing/maintenance_batch_*.csv")
    )

# ── Silver: CDC — apply turbine status changes as SCD Type 2 ─────────────────────
@dp.view
def turbine_cdc_feed():
    return spark.readStream.table("trainer_demo.demo_07.turbine_events")

dp.create_streaming_table("turbines_silver")

dp.create_auto_cdc_flow(
    target              = "turbines_silver",
    source              = "turbine_cdc_feed",
    keys                = ["turbine_id"],
    sequence_by         = col("sequence_num"),
    apply_as_deletes    = col("operation") == 'DELETE',
    except_column_list  = ["operation", "sequence_num"],
    stored_as_scd_type  = 2
)

In [ ]:
-- === PIPELINE SOURCE CODE (SQL equivalent) ===
-- Attach this notebook to a Lakeflow Pipeline

-- Bronze: stream new CSV files from landing volume (Auto Loader enabled via STREAM keyword)
CREATE OR REFRESH STREAMING TABLE maintenance_bronze_sql
AS SELECT *
FROM STREAM read_files(
    '/Volumes/trainer_demo/demo_07/sensor_data_landing/maintenance_batch_*.csv',
    format => 'csv',
    header => true
);

-- Silver: apply CDC turbine status changes as SCD Type 2
CREATE OR REFRESH STREAMING TABLE turbines_silver_sql;

CREATE FLOW turbines_cdc_flow
AS AUTO CDC INTO turbines_silver_sql
FROM STREAM(trainer_demo.demo_07.turbine_events)
KEYS (turbine_id)
APPLY AS DELETE WHEN operation = 'DELETE'
SEQUENCE BY sequence_num
COLUMNS * EXCEPT (operation, sequence_num)
STORED AS SCD TYPE 2;

#### Using multiple flows for regional data consolidation

GreenGrid receives turbine readings separately from each regional grid operator. **Explicit flows** merge them into a single gold-layer table without a `UNION` clause:

```sql
-- Target table for all regions combined
CREATE OR REFRESH STREAMING TABLE turbine_readings_global;

-- Each regional flow appends independently
CREATE FLOW readings_europe_flow
AS INSERT INTO turbine_readings_global BY NAME
SELECT *, 'Europe' AS region
FROM STREAM(bronze_readings_europe);

CREATE FLOW readings_north_america_flow
AS INSERT INTO turbine_readings_global BY NAME
SELECT *, 'North America' AS region
FROM STREAM(bronze_readings_north_america);
```

This pattern is more flexible than `UNION` — you can add a new region by adding a new flow without modifying or restarting existing flows.